In [17]:
import os
import pandas as pd
from datetime import datetime
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

### Aunxiliary functions

In [10]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\n{model_name}:")
    print(f"MSE: {mse:.2f}")
    print(f"R2 Score: {r2:.2f}")

    for feature, coef in zip(X.columns, model.coef_):
      print(f"{feature}: {coef:.4f}")

    return model, y_pred

### Read Data

In [2]:
BASE_PATH = os.path.abspath('..\data')
df = pd.read_csv(os.path.join(BASE_PATH, 'silver', 'joined_purchases_prep.csv'))
df['date'] = df['date'].apply(
    lambda x: datetime.strptime(str(x), '%Y-%m-%d')
)
df.head()

,date,productcategory,company_type,employees,quantity,unemployment_rate,gdp_value,mortgage_rate_30y,cpi,quantity_shifted
0,2020-02-01,Appliances,non-profit,1-10,889.0,3.5,21481.367,3.51,257.971,610.0
1,2020-02-01,Appliances,non-profit,101-500,648.0,3.5,21481.367,3.51,257.971,1544.0
2,2020-02-01,Appliances,private,1-10,1862.0,3.5,21481.367,3.51,257.971,2535.0
3,2020-02-01,Appliances,private,"500-1,000",785.0,3.5,21481.367,3.51,257.971,923.0
4,2020-02-01,Appliances,private,51-100,3171.0,3.5,21481.367,3.51,257.971,3704.0


In [5]:
# Extract month as an additional feature
df['month'] = df['date'].apply(lambda x: str(x.month))
df.head()

,date,productcategory,company_type,employees,quantity,unemployment_rate,gdp_value,mortgage_rate_30y,cpi,quantity_shifted,month
0,2020-02-01,Appliances,non-profit,1-10,889.0,3.5,21481.367,3.51,257.971,610.0,2
1,2020-02-01,Appliances,non-profit,101-500,648.0,3.5,21481.367,3.51,257.971,1544.0,2
2,2020-02-01,Appliances,private,1-10,1862.0,3.5,21481.367,3.51,257.971,2535.0,2
3,2020-02-01,Appliances,private,"500-1,000",785.0,3.5,21481.367,3.51,257.971,923.0,2
4,2020-02-01,Appliances,private,51-100,3171.0,3.5,21481.367,3.51,257.971,3704.0,2


### Split data and scale features

In [7]:
X = df.drop(
    ['date', 'quantity_shifted'],
    axis=1
)

y = df.quantity_shifted

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2520, 9)
(630, 9)
(2520,)
(630,)


In [28]:
# Transform and scale features

# Custom transformer to simulate one hot encoding
categorical_cols = ['productcategory', 'company_type', 'employees']
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(X_train[categorical_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)
encoded_df.head()

,productcategory_Appliances,productcategory_Audio,productcategory_Automotive,productcategory_Books & E-readers,productcategory_Cameras,productcategory_Clothing,productcategory_Computers,productcategory_Cookware,productcategory_Electronics,productcategory_Fitness & Health,...,productcategory_Sports & Outdoors,productcategory_Toys & Games,productcategory_Wearable Tech,company_type_non-profit,company_type_private,company_type_public,employees_1-10,employees_101-500,"employees_500-1,000",employees_51-100
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
